# 10b · Spam en español por traducción automática (OPUS-MT) — Grupo 17 · vía B
**GPU recomendada** (Colab T4: ~23k correos en 25–40 min; en CPU tardaría horas). Reanudable: guarda por lotes y continúa.

Cubre la clase spam, ausente en los corpus nativos. Traduce el spam en inglés ya validado (CEAS + SpamAssassin de `emails_v1`)
con `Helsinki-NLP/opus-mt-en-es` (CC BY 4.0). Para que el "estilo de traducción" no quede ligado solo a la clase spam,
traduce además una muestra de phishing y legítimo (control de artefacto). La vía 3 (spam sintético) queda como celda OPCIONAL y declarada.

**Criterios de cierre:** (1) spam_es con su origen y hash; (2) muestra de control traducida; (3) 0 IOCs vivos;
(4) `emails_es_v1` ampliado con la clase spam; (5) chequeo de artefacto de traducción (¿un modelo distingue traducido vs nativo?).

In [1]:
# Celda 1: Entorno
import os, sys, json, hashlib, random, importlib.util, time
from pathlib import Path
import numpy as np, pandas as pd
SEED = 17; random.seed(SEED); np.random.seed(SEED)
def detectar_entorno():
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"): return "kaggle"
    try:
        import google.colab  # noqa
        return "colab"
    except ImportError:
        return "local"
ENV = detectar_entorno()
if ENV == "colab":
    from google.colab import drive; drive.mount("/content/drive"); BASE = Path("/content/drive/MyDrive/PI_G17")
else:
    BASE = Path(os.environ.get("PI_G17_BASE", Path.cwd() / "PI_G17"))
PROC, INTERIM, RES = BASE / "data/processed", BASE / "data/interim", BASE / "results"
INTERIM.mkdir(parents=True, exist_ok=True)
def cargar(nombre, ruta):
    spec = importlib.util.spec_from_file_location(nombre, ruta); m = importlib.util.module_from_spec(spec)
    sys.modules[nombre] = m; spec.loader.exec_module(m); return m
ioc = cargar("ioc_utils", BASE / "src/ioc_utils.py"); dp = cargar("data_prep", BASE / "src/data_prep.py")
assert (PROC / "emails_v1.parquet").exists() and (PROC / "emails_es_v1.parquet").exists(), "Faltan emails_v1 (03) o emails_es_v1 (10)"
LOTE_PARQUET = INTERIM / "spam_es_traducido.parquet"
print(ENV, BASE)

Mounted at /content/drive
colab /content/drive/MyDrive/PI_G17


In [2]:
# Celda 2: Dependencias y modelo de traducción
%pip install -q "transformers>=4.40" sentencepiece sacremoses torch
import torch
from transformers import MarianMTModel, MarianTokenizer
DEV = "cuda" if torch.cuda.is_available() else "cpu"
MODELO_MT = "Helsinki-NLP/opus-mt-en-es"
tok_mt = MarianTokenizer.from_pretrained(MODELO_MT)
mt = MarianMTModel.from_pretrained(MODELO_MT).to(DEV).eval()
print("traductor:", MODELO_MT, "| dispositivo:", DEV)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 867.8/867.8 kB 2.6 MB/s eta 0:00:00


tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/826k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.59M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  312MB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

traductor: Helsinki-NLP/opus-mt-en-es | dispositivo: cuda


In [3]:
# Celda 3: Traductor robusto: defang -> traducir por frases -> re-defang
# Se traduce el texto YA defangeado (ioc_utils). Los marcadores hxxp[://], __url__, etc. no deben romperse:
# se protegen como tokens que MarianMT no altera, se traduce, y se restauran. Trunca a 400 palabras/correo (spam largo).
import re
PROT = re.compile(r"(hxxps?\[://\][^\s]*|fxp\[://\][^\s]*|www\[\.\][^\s]*|\b\d{1,3}\[\.\]\d{1,3}\[\.\]\d{1,3}\[\.\]\d{1,3}\b|__[a-z]+__)")
def proteger(t):
    fichas = {}
    def sub(m):
        k = f"ZQX{len(fichas)}QZ"; fichas[k] = m.group(0); return k
    return PROT.sub(sub, t), fichas
def restaurar(t, fichas):
    for k, v in fichas.items(): t = t.replace(k, v)
    return t
@torch.no_grad()
def traducir_lote(textos, bs=32, max_pal=400):
    salida = []
    protegidos = [proteger(" ".join((t or "").split()[:max_pal])) for t in textos]
    solo_texto = [p[0] for p in protegidos]
    for i in range(0, len(solo_texto), bs):
        chunk = solo_texto[i:i + bs]
        enc = tok_mt(chunk, return_tensors="pt", padding=True, truncation=True, max_length=512).to(DEV)
        gen = mt.generate(**enc, max_length=512, num_beams=2)
        salida.extend(tok_mt.batch_decode(gen, skip_special_tokens=True))
    return [restaurar(s, protegidos[i][1]) for i, s in enumerate(salida)]
# prueba mínima de protección
_t = "verifica en hxxps[://]banco[.]example[.]com y el marcador __url__ sigue intacto"
print(traducir_lote([_t])[0])

verifica en hxxps[://]banco[.]example[.]com y el marcador __url__ sigue intacto


In [4]:
# Celda 4: Material a traducir: todo el spam + muestra de control
V1 = pd.read_parquet(PROC / "emails_v1.parquet", columns=["id", "clase", "grupo", "texto_modelo"])
V0 = pd.read_parquet(INTERIM / "emails_v0.parquet", columns=["hash", "texto_defang"]) if (INTERIM / "emails_v0.parquet").exists() else None
# texto crudo defangeado (mejor para traducir que el ya marcado); si no está, se usa texto_modelo
fuente_txt = "texto_defang" if V0 is not None else "texto_modelo"
base = V1.merge(pd.read_parquet(PROC/"emails_v1.parquet", columns=["id","hash"]), on="id").merge(V0, on="hash", how="left") if V0 is not None else V1.assign(texto_defang=V1["texto_modelo"])
SPAM = base[base["clase"] == "spam"].copy()
# v2: para romper el artefacto (AUC 0,999) NO se traduce el spam en español -> en su lugar se construye un espejo:
# se traduce al español TODO el phishing y legítimo en inglés, de modo que las 3 clases del set TRADUCIDO comparten
# el mismo "estilo de traducción". El español nativo (SpaPhish/SpearPhishMX) se usa aparte en la evaluación.
NO_SPAM = base[base["clase"] != "spam"]
A_TRADUCIR = pd.concat([SPAM.assign(rol="spam"), NO_SPAM.assign(rol="espejo")], ignore_index=True)
print("spam:", len(SPAM), "| espejo no-spam:", len(NO_SPAM), "| total a traducir:", len(A_TRADUCIR))

spam: 23562 | espejo no-spam: 32822 | total a traducir: 56384


In [5]:
# Celda 5: Traducción por lotes, reanudable
HECHO = pd.read_parquet(LOTE_PARQUET) if LOTE_PARQUET.exists() else pd.DataFrame(columns=["id", "rol", "texto_es"])
hechos = set(HECHO["id"]); pend = A_TRADUCIR[~A_TRADUCIR["id"].isin(hechos)].reset_index(drop=True)
print(f"ya traducidos: {len(hechos)} | pendientes: {len(pend)}")
PASO = 512; t0 = time.time()
for i in range(0, len(pend), PASO):
    sub = pend.iloc[i:i + PASO]
    tr = traducir_lote(sub[fuente_txt].tolist())
    HECHO = pd.concat([HECHO, pd.DataFrame({"id": sub["id"].values, "rol": sub["rol"].values, "texto_es": tr})], ignore_index=True)
    HECHO.to_parquet(LOTE_PARQUET, index=False)           # checkpoint tras cada bloque
    print(f"  {len(HECHO)}/{len(A_TRADUCIR)}  ({time.time()-t0:.0f}s)")
print("traducción completa:", len(HECHO))

ya traducidos: 53686 | pendientes: 1106
  55234/56384  (303s)
  55746/56384  (606s)
  55828/56384  (655s)
traducción completa: 55828


In [6]:
# Celda 6: (OPCIONAL, vía 3) spam sintético — DESACTIVADO por defecto
# Requiere un endpoint LLM; se deja declarado y reproducible. Para activarlo: USAR_SINTETICO=True y definir generar().
USAR_SINTETICO = False
sinteticos = pd.DataFrame(columns=["id", "texto_es"])
if USAR_SINTETICO:
    CATEGORIAS = ["publicidad agresiva", "productos milagro", "criptomonedas", "cadena de la suerte",
                  "oferta de empleo falsa", "préstamo instantáneo", "premio/lotería"]
    def generar(categoria, n):
        raise NotImplementedError("Conectar aquí un LLM (API o local). Devolver lista de n correos de spam en español.")
    filas = []
    for cat in CATEGORIAS:
        for j, t in enumerate(generar(cat, 150)):
            filas.append({"id": f"syn_{cat[:4]}_{j}", "texto_es": t})
    sinteticos = pd.DataFrame(filas)
print("sintéticos:", len(sinteticos), "(vía 3; declarada, desactivada por defecto)")

sintéticos: 0 (vía 3; declarada, desactivada por defecto)


In [7]:
# Celda 7: Construir spam_es y ampliar emails_es_v1
from sklearn.model_selection import train_test_split
def fila_es(texto_es, idx, grupo):
    td = ioc.defang_texto(texto_es)
    return {"id": idx, "fuente": grupo, "grupo": grupo, "clase": "spam", "y": 1,  # clase/y se sobre-escriben en fila_es2 "anio": None,
            "hash": hashlib.sha1(dp.texto_plano(td).lower().encode("utf-8","replace")).hexdigest(),
            "texto_defang": td, "texto_modelo": dp.texto_para_modelo(td)}
# Espejo traducido completo: cada correo conserva SU clase original (no solo spam)
clase_de = dict(zip(base["id"], base["clase"])); y_de = {"legitimo": 0, "spam": 1, "phishing": 2}
def fila_es2(texto_es, idx_orig):
    c = clase_de[idx_orig]; r = fila_es(texto_es, f"tr_{idx_orig}"[:18], f"traducido_{c}")
    r["clase"], r["y"] = c, y_de[c]; return r
filas = [fila_es2(t, i) for i, t in zip(HECHO["id"], HECHO["texto_es"])]
if len(sinteticos):
    filas += [fila_es(t, sid, "spam_es_sintetico") for sid, t in zip(sinteticos["id"], sinteticos["texto_es"])]
S = pd.DataFrame(filas)
S["n_urls"] = S["texto_defang"].map(lambda t: len(ioc.urls_para_features(t))); S["sin_url"] = S["n_urls"] == 0
S["es_html"] = False; S["len_plano"] = S["texto_modelo"].str.len(); S["urls_defang"] = S["texto_defang"].map(lambda t: ioc.extraer_iocs(t)["urls"])
S = S[(S["len_plano"] >= 20) & ~S.duplicated("hash")].copy()
assert not S["texto_modelo"].map(ioc.iocs_vivos).map(bool).any()
estr = S["grupo"]
tr, tmp = train_test_split(S.index, test_size=0.30, stratify=estr, random_state=SEED)
va, te = train_test_split(tmp, test_size=0.50, stratify=estr.loc[tmp], random_state=SEED)
S["split"] = "train"; S.loc[va, "split"] = "val"; S.loc[te, "split"] = "test"
ES = pd.read_parquet(PROC / "emails_es_v1.parquet")
fold_cols = [c for c in ES.columns if c.startswith(("fold_", "fuera_"))]
for c in fold_cols:
    S[c] = S["split"] if c.startswith("fold_") else False      # el spam_es nunca es "lo fuera" en los folds de phishing nativo
for c in ES.columns:
    if c not in S.columns: S[c] = None
ES2 = pd.concat([ES, S[ES.columns]], ignore_index=True)
ES2.to_parquet(PROC / "emails_es_v2.parquet", index=False)
TABLA = pd.crosstab(ES2["grupo"], ES2["clase"], margins=True).to_dict()
print("emails_es_v2:", len(ES2), "correos"); TABLA

/tmp/ipykernel_648/3463156313.py:31: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  ES2 = pd.concat([ES, S[ES.columns]], ignore_index=True)


emails_es_v2: 55130 correos


{'legitimo': {'SpaPhish': 632,
  'SpearPhishMX': 1059,
  'traducido_legitimo': 21083,
  'traducido_phishing': 0,
  'traducido_spam': 0,
  'All': 22774},
 'phishing': {'SpaPhish': 711,
  'SpearPhishMX': 1866,
  'traducido_legitimo': 0,
  'traducido_phishing': 9753,
  'traducido_spam': 0,
  'All': 12330},
 'spam': {'SpaPhish': 0,
  'SpearPhishMX': 0,
  'traducido_legitimo': 0,
  'traducido_phishing': 0,
  'traducido_spam': 20026,
  'All': 20026},
 'All': {'SpaPhish': 1343,
  'SpearPhishMX': 2925,
  'traducido_legitimo': 21083,
  'traducido_phishing': 9753,
  'traducido_spam': 20026,
  'All': 55130}}

In [8]:
# Celda 8: Chequeo de artefacto de traducción
# ¿Un clasificador distingue "traducido" de "nativo"? Si el AUC es alto, el estilo de traducción es separable (se declara).
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
nativo = pd.read_parquet(PROC / "emails_es_v1.parquet", columns=["texto_modelo"]).assign(trad=0)
trad = S[S["grupo"].str.startswith("traducido_")][["texto_modelo"]].assign(trad=1)
M = pd.concat([nativo, trad], ignore_index=True).sample(frac=1, random_state=SEED)
n = min((M["trad"] == 0).sum(), (M["trad"] == 1).sum())
M = pd.concat([M[M.trad == 0].head(n), M[M.trad == 1].head(n)])
Xtr, Xte, ytr, yte = train_test_split(M["texto_modelo"], M["trad"], test_size=0.3, stratify=M["trad"], random_state=SEED)
vec = TfidfVectorizer(max_features=20000, ngram_range=(1,2)); clf = LogisticRegression(max_iter=1000)
clf.fit(vec.fit_transform(Xtr), ytr)
AUC_ARTEFACTO = round(roc_auc_score(yte, clf.predict_proba(vec.transform(Xte))[:,1]), 3)
print("AUC traducido-vs-nativo:", AUC_ARTEFACTO, "(cercano a 0.5 = indistinguible; alto = artefacto separable)")

AUC traducido-vs-nativo: 0.997 (cercano a 0.5 = indistinguible; alto = artefacto separable)


In [9]:
# Celda 9: RESUMEN
RESUMEN = {"entorno": ENV, "traductor": MODELO_MT, "spam_traducido": int((HECHO["rol"]=="spam").sum()),
           "control_traducido": int((HECHO["rol"]=="control").sum()), "sinteticos": int(len(sinteticos)),
           "spam_es_final": int((ES2["grupo"].isin(["spam_es_traducido","spam_es_sintetico"])).sum()),
           "emails_es_v2_grupo_x_clase": TABLA, "auc_artefacto_traduccion": AUC_ARTEFACTO,
           "ejemplo_traducido": ioc.defang_texto(HECHO[HECHO.rol=="spam"]["texto_es"].iloc[0][:200]) if (HECHO.rol=="spam").any() else None}
(RES / "spam_es_v1.json").write_text(json.dumps(RESUMEN, indent=1, ensure_ascii=False, default=str))
print("===== RESUMEN (10b) =====")
print(json.dumps(RESUMEN, ensure_ascii=False, default=str, separators=(",", ":")))

===== RESUMEN (10b) =====
{"entorno":"colab","traductor":"Helsinki-NLP/opus-mt-en-es","spam_traducido":23562,"control_traducido":3000,"sinteticos":0,"spam_es_final":0,"emails_es_v2_grupo_x_clase":{"legitimo":{"SpaPhish":632,"SpearPhishMX":1059,"traducido_legitimo":21083,"traducido_phishing":0,"traducido_spam":0,"All":22774},"phishing":{"SpaPhish":711,"SpearPhishMX":1866,"traducido_legitimo":0,"traducido_phishing":9753,"traducido_spam":0,"All":12330},"spam":{"SpaPhish":0,"SpearPhishMX":0,"traducido_legitimo":0,"traducido_phishing":0,"traducido_spam":20026,"All":20026},"All":{"SpaPhish":1343,"SpearPhishMX":2925,"traducido_legitimo":21083,"traducido_phishing":9753,"traducido_spam":20026,"All":55130}},"auc_artefacto_traduccion":0.997,"ejemplo_traducido":"Olvidado de peso Buenos días, amigo. Pérdida de peso sin tener hambre! Pure Hoodia proporciona un buen enfoque natural para la pérdida de peso. Hoodia ha sido presentado en CNN, BBC, OPRAH y CBS 60 Mi"}
